In [0]:
%restart_python


In [0]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'src'))


In [0]:
from config import ProjectConfig
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
config = ProjectConfig.from_yaml(config_path="../project_config_telcochurn.yml", env="dev")
dataset_path = f"/Volumes/workspace/telco/telco_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [0]:
from data_processing import DataProcessing

In [0]:
# Execução do pipeline
data_processing = DataProcessing(dataset_path, config, spark)
data_processing.preprocess()
df_train, df_test = data_processing.split_data(train_size=0.8, seed=42)
data_processing.save_feature_tables(df_train, df_test)

In [0]:
%pip install loguru


In [0]:
import mlflow
import pandas as pd
from delta.tables import DeltaTable
from lightgbm import LGBMClassifier
from loguru import logger
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pyspark.sql import SparkSession
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from marvel_characters.config import ProjectConfig, Tags

In [0]:
import warnings
import os

# Silencia avisos do sistema e bibliotecas compiladas
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# ==========================================
# 1. PARAMETRIZAÇÃO DAS COLUNAS (LIMPAS)
# ==========================================
features_categoricas = [
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "MultipleLines",
    "TechSupport",
    "Contract_OHE_Month_to_month",
    "Contract_OHE_One_year",
    "Contract_OHE_Two_year"
]

features_numericas = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

coluna_target = "Churn"

# ==========================================
# 2. CARREGAMENTO E CONVERSÃO DOS DADOS
# ==========================================
df_train_spark = df_train
df_test_spark = df_test

train_pd = df_train_spark.toPandas()
test_pd = df_test_spark.toPandas()

# ==========================================
# 3. SEPARAÇÃO DE FEATURES (X) E TARGET (y)
# ==========================================
X_train = train_pd[features_categoricas + features_numericas]
X_test = test_pd[features_categoricas + features_numericas]

y_train = train_pd[coluna_target]
y_test = test_pd[coluna_target]

# ==========================================
# 4. TREINAMENTO E AVALIAÇÃO - LIGHTGBM
# ==========================================
lgb_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    random_state=42,
    scale_pos_weight=2.5,
    n_jobs=-1,
    verbose=-1 # Silencia logs internos do LightGBM
)
lgb_model.fit(X_train, y_train)

lgb_pred = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("             MÉTRICAS - LIGHTGBM")
print("="*50)
print(classification_report(y_test, lgb_pred))
print(f"AUC-ROC Score: {roc_auc_score(y_test, lgb_proba):.4f}")
print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, lgb_pred))
print("="*50)

In [0]:
import optuna
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score

# Silenciar logs excessivos do Optuna e LightGBM
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 1. DEFINIÇÃO DA FUNÇÃO OBJETIVO (FOCUS: F1-SCORE)
# ==========================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 4.0), # Ampliado para ajudar no F1
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1
    }
    
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    
    # Nova métrica alvo: Equilíbrio Harmônico (F1-Score) na classe 1 (Churn)
    f1 = f1_score(y_test, preds, pos_label=1.0, zero_division=0)
    
    return f1

# ==========================================
# 2. EXECUÇÃO DA OTIMIZAÇÃO
# ==========================================
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# ==========================================
# 3. TREINAMENTO DO MODELO CAMPEÃO
# ==========================================
print("="*50)
print("             RESULTADOS DO TUNING (F1)")
print("="*50)
print(f"Melhor F1-Score alcançado: {study.best_value:.4f}")
print("\nMelhores hiperparâmetros encontrados:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")
print("="*50)

best_params = study.best_params
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1

final_lgb_model = LGBMClassifier(**best_params)
final_lgb_model.fit(X_train, y_train)

# ==========================================
# 4. AVALIAÇÃO DO MODELO OTIMIZADO
# ==========================================
final_preds = final_lgb_model.predict(X_test)
final_proba = final_lgb_model.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("         MÉTRICAS DO LIGHTGBM OTIMIZADO (F1)")
print("="*50)
print(classification_report(y_test, final_preds))
print(f"AUC-ROC Score Final: {roc_auc_score(y_test, final_proba):.4f}")
print("\nMatriz de Confusão Final:")
print(confusion_matrix(y_test, final_preds))
print("="*50)

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient 
fe = FeatureEngineeringClient()
# create a feature table from the dataset
table_name = f"mlops_dev.telco_churn.telco_customer_features"

fe.create_table(
    name=table_name,
    primary_keys=["customerID"],
    df=df,
    #partition_columns=["InternetService"] for small datasets partitioning is not recommended
    description="Telco customer features",
    tags={"source": "bronze", "format": "delta"}
)

In [0]:
df.printSchema()

In [0]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 1. Converte o seu DataFrame do Spark para Pandas
# Se os seus dados já estiverem em Pandas, pode ignorar esta linha
df_pandas = df.toPandas()

# 2. Define as colunas que serão codificadas
categorical_cols = ["Contract", "PaymentMethod"]

# 3. Configura o codificador OneHot
# sparse_output=False garante que o resultado seja uma matriz densa (vetor numérico comum)
oh_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# 4. Cria o preprocessador apenas para as colunas selecionadas
# remainder="passthrough" mantém as outras colunas do DataFrame intactas
preprocessor = ColumnTransformer(
    transformers=[
        ("onehot", oh_encoder, categorical_cols)
    ],
    remainder="passthrough"
)

# 5. Executa a transformação
encoded_matrix = preprocessor.fit_transform(df_pandas[categorical_cols])

# 6. Salva a matriz de vetores dentro de uma nova coluna chamada 'features'
df_pandas["features"] = list(encoded_matrix)

# Exibe o resultado final com a nova coluna de vetores
df_pandas[["Contract", "PaymentMethod", "features"]].head()

In [0]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# 1. Convertendo o DataFrame para Pandas para simular o seu ambiente
# (Se o seu df já for Pandas, ignore esta linha)
sample_pdf = df.select("Contract").distinct().toPandas()

string_cols = ["Contract"]

# 2. Configurando o Pipeline de Transformação (Equivalente ao StringIndexer + OneHotEncoder)
# O OneHotEncoder do scikit-learn pode receber strings diretamente, mas configuramos
# sparse_output=False para gerar uma matriz densa (equivalente ao vetor denso do VectorAssembler)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", encoder, string_cols)
    ],
    remainder="passthrough"
)

# 3. Fit e Transform nos dados
features_array = preprocessor.fit_transform(sample_pdf)

# 4. Criando o DataFrame de resultado para exibição
# O Scikit-learn retorna um array NumPy, então o transformamos de volta para Pandas
encoded_cols = preprocessor.named_transformers_["cat"].get_feature_names_out(string_cols)

result_df_display = sample_pdf.copy()
# Armazena o array/vetor denso em uma única coluna para espelhar o comportamento do VectorAssembler
result_df_display["features"] = list(features_array)

# Exibe o resultado final
print(result_df_display[["Contract", "features"]])

In [0]:
from pyspark.ml.feature import OneHotEncoder


# Create a list of one-hot encoded feature names
ohe_cols = [column + "_ohe" for column in string_cols]

# Instantiate the OneHotEncoder with the column lists
ohe = OneHotEncoder(inputCols=index_cols, outputCols=ohe_cols, handleInvalid="keep")

# Fit the OneHotEncoder on the indexed data
ohe_model = ohe.fit(indexed_df)

# Transform indexed_df using the ohe_model
ohe_df = ohe_model.transform(indexed_df)
ohe_df.show()

In [0]:
data_processing = DataProcessing(dataset_path, config, spark)
# data_processing.preprocess()
data_processing.preprocess()

In [0]:
import time
import numpy as np
import pandas as pd
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, to_utc_timestamp, col, count, when, avg
from pyspark.sql.types import BooleanType, ShortType, IntegerType, DoubleType, StringType
from sklearn.model_selection import train_test_split
from data_utils import calculate_missing
from pyspark.sql import SparkSession



class DataProcessing:

    # 1. Altere o tipo do parâmetro para str (caminho do dataset)
    def __init__(self, dataset_path: str, config: ProjectConfig, spark: SparkSession) -> None:
        # 2. Defina o self.spark PRIMEIRO
        self.spark = spark 
        self.config = config
        # 3. Agora self.spark está disponível para fazer a leitura
        self.df = (
            self.spark.read
            .option("nullValue", " ")
            .csv(dataset_path, header=True, multiLine=True, escape="'", inferSchema=True)
        )
    def preprocess(self) -> None:
        cat_features = self.config.cat_features
        num_features = self.config.num_features
        binary_features = self.config.binary_features
        target = self.config.target
        per_thresh = 0.6

        #Senior_Citizen
        self.df = self.df.withColumn("SeniorCitizen", col("SeniorCitizen").cast(IntegerType()))
        self.df = self.df.withColumn("SeniorCitizen", when(col("SeniorCitizen") == 1, True).otherwise(False))

        # Tenure months to Long/Integer 
        self.df = self.df.withColumn("tenure", col("tenure").cast(DoubleType()))

        #Binary features to boolean
        for feature in binary_features:
            self.df = self.df.withColumn(feature, col(feature).cast(BooleanType()))

        #Removing outliers from Payment Method

        TotalCharges_cutoff = 0

        # Use .filter method and SQL col() function
        telco_no_outliers_df = self.df.filter(\
            (col("TotalCharges") > TotalCharges_cutoff) | \
            (col("TotalCharges").isNull())) # Keep Nulls

        group_var = "PaymentMethod"
        stats_df = telco_no_outliers_df.groupBy(group_var) \
                      .agg(count("*").alias("Total"),\
                           avg("MonthlyCharges").alias("MonthlyCharges")) \
                      .orderBy(col("Total").desc())

        N = telco_no_outliers_df.count()  # total count
        lower_groups = [elem[group_var] for elem in stats_df.head(2) if elem['Total']/N < 0.2 and elem['MonthlyCharges'] < 20]
        print(f"Removing groups: {','.join(lower_groups)}")

        self.df = telco_no_outliers_df.filter( \
            ~col(group_var).isin(lower_groups) | \
            col(group_var).isNull())
        
        '''Handling missing values'''
        missing_df = calculate_missing(self.df, self.spark)
        N = self.df.count()
        to_drop_missing = [x.asDict()['Column'] for x in missing_df.select("Column").where(col("Number of Missing Values") / N >= per_thresh).collect()]
        print(f"Dropping columns {to_drop_missing} for more than {per_thresh * 100}% missing data")
        self.df = self.df.drop(*to_drop_missing)

        #Imputing missing data

        num_cols = [c.name for c in self.df .schema.fields if (c.dataType == DoubleType() or c.dataType == IntegerType())]
        #FILL NULL WITH 0 FOR NUMERIC DATA
        self.df = self.df.na.fill(value=0, subset=num_cols)

        # Get a list of boolean columns
        bool_cols = [c.name for c in self.df.schema.fields if (c.dataType == BooleanType())]
        self.df = self.df.na.fill(value=False, subset=bool_cols)

        # Get list of string cols
        to_exclude = ["customerID", "gender", "Contract", "PaymentMethod"]
        string_cols = [c.name for c in self.df.drop(*to_exclude).schema.fields if c.dataType == StringType()]

        # Impute
        self.df = self.df.na.fill(value='No', subset=string_cols)

        #ENCODING CATEGORICAL FEATURE


        return self.df



data_processing = DataProcessing(dataset_path, config, spark)
data_processing.preprocess()

In [0]:
#from pyspark.sql.functions import NoneType
per_thresh = 0.6  # Drop if column has more than 80% missing data

N = df.count()  # total count
to_drop_missing = [x.asDict()['Column'] for x in missing_df.select("Column").where(col("Number of Missing Values") / N >= per_thresh).collect()]

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
#from data_processing import DataProcessing

In [0]:
# Execução
spark = SparkSession.builder.getOrCreate()
dataset_path = f"/Volumes/workspace/telco/telco_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = spark.read.option("nullValue", " ").csv(dataset_path, header=True, multiLine=True, escape="'", inferSchema=True)
display(df)

In [0]:
print(df.schema)

In [0]:
num_features = [
    "tenure", 
    "MonthlyCharges", 
    "TotalCharges"]

cat_features = [
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod'
]

binary_features = ["SeniorCitizen", "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]

In [0]:
# from pyspark.sql.functions import col, count

# for feature in cat_features:
#     df.groupBy(feature).agg(count(col(feature)).alias("total")).display()

In [0]:
from config import 

In [0]:
#from data_utils import calculate_missing
spark = SparkSession.builder.getOrCreate()

missing_df = calculate_missing(df, spark)

In [0]:
from pyspark.sql.types import BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, when, lower, trim

df_01 = df

for feature in binary_features:
    # Verifica o tipo da coluna no schema
    feature_type = df_01.schema[feature].dataType

    # Só faz cast para string se a coluna não for StringType
    if isinstance(feature_type, StringType):
        feature_clean = lower(trim(col(feature)))
    else:
        feature_clean = lower(trim(col(feature).cast("string")))

    df_01 = df_01.withColumn(
        feature,
        when(feature_clean.isin("yes", "1", "true"), 1)
        .when(feature_clean.isin("no", "0", "false"), 0)
        .otherwise(None)
        .cast(DoubleType())
    )

display(df_01.select(binary_features + num_features))

In [0]:
df_01.select(num_features).summary(
    "count",
    "mean",
    "stddev",
    "min",
    "1%",
    "5%",
    "25%",
    "50%",
    "75%",
    "95%",
    "99%",
    "max"
).display()

In [0]:
from pyspark.sql.functions import col

feature = "TotalCharges"

q1, q3 = df_01.approxQuantile(feature, [0.25, 0.75], 0.01)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Feature:", feature)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Limite inferior:", lower_bound)
print("Limite superior:", upper_bound)

outliers_count = df_01.filter(
    (col(feature) < lower_bound) |
    (col(feature) > upper_bound)
).count()

total_count = df_01.count()

print("Quantidade de outliers:", outliers_count)
print("Percentual de outliers:", round(outliers_count / total_count * 100, 2), "%")

df_no_outliers = df_01.filter(
    (col(feature) >= lower_bound) &
    (col(feature) <= upper_bound)
)

print("Linhas antes:", total_count)
print("Linhas depois:", df_no_outliers.count())

display(df_no_outliers)